<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller: Sombras, Hojas y la Hormiga en la Alfombra 🔦🐜 🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Taller Práctico Evaluativo Para Dummies · Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/Para%20Dummies/05_Reduccion_Dimensionalidad_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

💡 En este taller vas a **resumir datos con menos columnas** sin perder lo importante, usando las ideas del módulo: la **sombra** que más estira los datos (PCA) y la **hormiga** que camina por una alfombra enrollada (Isomap). No hay respuestas puestas: completas los huecos `...`, pero con **pistas** en cada paso. Al terminar sabrás:

1. 🔦 **Encontrar la mejor sombra** de una nube de puntos y ver cuánta información conserva.
2. 🎒 **Decidir cuántas sombras guardar** (cuántas columnas nuevas) para conservar el 95 % de la información.
3. 🐜 **Elegir entre PCA e Isomap** cuando los datos son una alfombra enrollada.

> 🧸 Si algo se traba, vuelve a [*PCA Para Dummies*](../../05%20-%20Reduccion%20de%20Dimensionalidad/Para%20Dummies/00_PCA_Dummies.ipynb) y a [*Isomap Para Dummies*](../../05%20-%20Reduccion%20de%20Dimensionalidad/Para%20Dummies/02_Isomap_Variedades_Dummies.ipynb). Aun así, **tú** resuelves el taller.

---

## Instrucciones

1. Reemplaza cada `...` por tu código (las pistas en los comentarios `# TODO` te dicen cómo).
2. Debajo de cada reto hay una celda de **autoverificación**: si algo falta, te dirá **qué revisar** con un mensaje amable; cuando todo esté bien verás un ✅.
3. No modifiques las celdas de autoverificación.
4. Antes de entregar usa **Kernel → Restart & Run All** y confirma que no quedan errores ni `...` sin completar.

---
## Preparación 🧰

Ejecuta esta celda tal como está. Crea tres conjuntos de datos pequeños, sin descargar nada:

- `hojas`: largo y ancho (cm) de 200 hojas, que forman una nube alargada y **inclinada**.
- `flores`: el clásico *Iris* (150 flores, 4 medidas), ya **estandarizado** (`flores_z`).
- `alfombra` / `pos`: 800 puntos 3D de una alfombra enrollada; `pos` es la **posición real** de cada punto a lo largo de la alfombra (la respuesta que queremos recuperar).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

SEED = 42
rng = np.random.default_rng(SEED)


def listo(*valores):
    """True si ninguna variable sigue siendo None o ... (sin completar)."""
    return all(v is not None and v is not Ellipsis for v in valores)


# Hojas: nube alargada e inclinada 30 grados (largo ~ 3 cm de variacion, ancho ~ 0.5 cm)
ang = np.deg2rad(30)
giro = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
hojas = (rng.normal(size=(200, 2)) * [3, 0.5]) @ giro.T + [12, 6]

# Flores de Iris estandarizadas
flores_z = StandardScaler().fit_transform(load_iris().data)

# Alfombra enrollada con su posicion real
alfombra, pos = make_swiss_roll(n_samples=800, noise=0.02, random_state=SEED)


def acuerdo(Y, pos):
    """Que tan bien algun eje de Y ordena los puntos como la posicion real (0 a 1)."""
    return max(abs(spearmanr(Y[:, j], pos)[0]) for j in range(Y.shape[1]))


plt.scatter(*hojas.T, s=10); plt.axis("equal"); plt.title("Las hojas"); plt.show()

---
## Reto 1: La mejor sombra de las hojas 🔦 (30 puntos)

Imagina que iluminas la nube de `hojas` desde un ángulo y miras la sombra sobre una **recta**. La mejor sombra es la que **más estira** los puntos (la que conserva más información). PCA la encuentra sola.

**Qué hacer**

1. Ajusta `PCA(n_components=1)` con `hojas` y guarda el modelo en `pca1`.
2. Guarda en `direccion` la dirección de la mejor sombra (`pca1.components_[0]`) y en `razon` la fracción de información que conserva (`explained_variance_ratio_[0]`).
3. Calcula `sombra` (la proyección de las hojas sobre la recta: `transform`) y `reconstruidas` (volver a 2D con `inverse_transform`).
4. Calcula `error_medio`: la distancia media entre cada hoja original y su reconstrucción.

**Para aprobar:** la dirección apunta en la inclinación real de la nube (30°), la sombra conserva más del 90 % de la información y el error medio es menor que 0.6 cm.

In [ ]:
# TODO 1: Pista -> PCA(n_components=..., ).fit(...)
pca1 = ...

# TODO 2: pistas -> pca1.components_[0]  y  pca1.explained_variance_ratio_[0]
direccion = ...
razon = ...

# TODO 3: pistas -> pca1.transform(hojas)  y  pca1.inverse_transform(sombra)
sombra = ...
reconstruidas = ...

# TODO 4: distancia entre cada hoja y su reconstruccion, y luego el promedio.
#   Pista: np.linalg.norm(hojas - reconstruidas, axis=1) da las 200 distancias
error_medio = ...

In [ ]:
# ---- Autoverificacion del Reto 1 (no modificar) ----
assert listo(pca1, direccion, razon, sombra, reconstruidas, error_medio), "Reto 1: todavia hay huecos '...' sin completar. Revisa los 4 pasos."
assert pca1.n_components == 1, "Reto 1: pide UNA sola sombra (n_components=1)."
assert abs(np.linalg.norm(direccion) - 1) < 1e-6, "Reto 1: 'direccion' debe ser el vector pca1.components_[0] (de largo 1)."
alineada = abs(np.dot(direccion, [np.cos(np.deg2rad(30)), np.sin(np.deg2rad(30))]))
assert alineada > 0.99, f"Reto 1: la direccion no apunta hacia la inclinacion de la nube (30 grados; alineacion {alineada:.2f}). Revisa que ajustaste con 'hojas'."
assert 0.9 < razon <= 1, f"Reto 1: la sombra deberia conservar mas del 90 % y 'razon' vale {razon:.2f}. Pista: explained_variance_ratio_[0]."
assert np.asarray(sombra).shape == (200, 1), f"Reto 1: 'sombra' debe tener forma (200, 1) y tiene {np.asarray(sombra).shape}. Pista: pca1.transform(hojas)."
assert np.asarray(reconstruidas).shape == (200, 2), "Reto 1: 'reconstruidas' debe volver a 2 columnas. Pista: pca1.inverse_transform(sombra)."
assert 0 < error_medio < 0.6, f"Reto 1: el error medio deberia estar entre 0 y 0.6 cm y te dio {error_medio:.2f}. Pista: promedia las 200 distancias con .mean()."
print(f"✅ Reto 1 correcto | la sombra conserva {razon:.1%} | error medio = {error_medio:.2f} cm")

---
## Reto 2: ¿Cuántas sombras guardo? 🎒 (30 puntos)

Las flores de Iris tienen **4 medidas**. ¿Cuántas columnas nuevas (sombras) bastan para conservar al menos el **95 %** de la información?

**Qué hacer**

1. Ajusta `PCA()` (sin límite de componentes) con `flores_z` y guarda en `var_acum` la información **acumulada** (`np.cumsum` de `explained_variance_ratio_`): 4 números que suben hasta 1.
2. Calcula `n95`: el **menor** número de sombras con `var_acum >= 0.95`. 💡 *Pista:* cuenta cuántas posiciones de `var_acum` están por debajo de 0.95 y súmale 1.
3. Ajusta `PCA(n_components=n95)` y guarda los puntos nuevos en `Z` (forma `(150, n95)`).
4. Guarda en `info_perdida` cuánta información **se pierde** al quedarte con `n95` sombras: `1 - var_acum[n95 - 1]`.

**Para aprobar:** el acumulado es coherente, `n95` es el menor posible, `Z` tiene la forma correcta y lo que se pierde es menos del 5 %.

In [ ]:
# TODO 1: Pista -> PCA().fit(flores_z).explained_variance_ratio_  y luego np.cumsum(...)
var_acum = ...

# TODO 2: Pista -> (var_acum < 0.95).sum() + 1
n95 = ...

# TODO 3: Pista -> PCA(n_components=n95).fit_transform(flores_z)
Z = ...

# TODO 4: lo que se pierde con n95 sombras
info_perdida = ...

In [ ]:
# ---- Autoverificacion del Reto 2 (no modificar) ----
assert listo(var_acum, n95, Z, info_perdida), "Reto 2: todavia hay huecos '...' sin completar. Revisa los 4 pasos."
assert len(var_acum) == 4, f"Reto 2: 'var_acum' debe tener 4 numeros (uno por sombra) y tiene {len(var_acum)}. Pista: PCA() sin n_components."
assert all(a <= b for a, b in zip(var_acum, var_acum[1:])) and abs(var_acum[-1] - 1) < 1e-9, "Reto 2: el acumulado debe subir hasta llegar a 1. Pista: np.cumsum(explained_variance_ratio_)."
assert var_acum[n95 - 1] >= 0.95, f"Reto 2: con {n95} sombras solo conservas {var_acum[n95 - 1]:.1%}, menos del 95 %."
assert n95 == 1 or var_acum[n95 - 2] < 0.95, f"Reto 2: con {n95 - 1} sombras ya llegabas al 95 %; busca el MENOR numero."
assert np.asarray(Z).shape == (150, n95), f"Reto 2: 'Z' debe tener forma (150, {n95}) y tiene {np.asarray(Z).shape}."
assert abs(info_perdida - (1 - var_acum[n95 - 1])) < 1e-9 and 0 <= info_perdida < 0.05, f"Reto 2: info_perdida debe ser 1 - var_acum[n95 - 1] (menos del 5 %); tu valor es {info_perdida:.3f}."
print(f"✅ Reto 2 correcto | basta con {n95} de 4 columnas | acumulado = {np.round(var_acum, 3)} | se pierde {info_perdida:.1%}")

---
## Reto 3: La hormiga y la alfombra enrollada 🐜 (30 puntos)

La `alfombra` está enrollada en 3D, pero cada punto tiene una **posición real** `pos` a lo largo de ella. Una buena representación 2D debería **ordenar los puntos como `pos`**. Probemos con la sombra recta (PCA) y con el camino de la hormiga (Isomap).

**Qué hacer**

1. Calcula `Y_pca = PCA(n_components=2).fit_transform(alfombra)`.
2. Calcula `Y_iso = Isomap(n_neighbors=10, n_components=2).fit_transform(alfombra)`.
3. Mide con la función `acuerdo(Y, pos)` (ya está en la preparación) cuánto recupera cada una la posición real: `acuerdo_pca` y `acuerdo_iso`.
4. Guarda en `ganador` el texto `"Isomap"` o `"PCA"` según cuál tenga mayor acuerdo (usa un `if`/`else`).

**Para aprobar:** Isomap recupera la posición real (acuerdo mayor que 0.8), PCA no (menor que 0.4) y el ganador es el correcto. 💡 *Moraleja:* una sombra recta no puede desenrollar una alfombra.

In [ ]:
# TODO 1: Pista -> PCA(n_components=2).fit_transform(...)
Y_pca = ...

# TODO 2: Pista -> Isomap(n_neighbors=10, n_components=2).fit_transform(...)
Y_iso = ...

# TODO 3: pista -> acuerdo(Y_pca, pos)  y  acuerdo(Y_iso, pos)
acuerdo_pca = ...
acuerdo_iso = ...

# TODO 4: escribe "Isomap" o "PCA" segun cual tenga mayor acuerdo
ganador = ...

In [ ]:
# ---- Autoverificacion del Reto 3 (no modificar) ----
assert listo(Y_pca, Y_iso, acuerdo_pca, acuerdo_iso, ganador), "Reto 3: todavia hay huecos '...' sin completar. Revisa los 4 pasos."
assert np.asarray(Y_pca).shape == (800, 2) and np.asarray(Y_iso).shape == (800, 2), "Reto 3: cada mapa debe tener 800 filas y 2 columnas. Pista: n_components=2 y ajusta con 'alfombra'."
assert abs(acuerdo_pca - acuerdo(np.asarray(Y_pca), pos)) < 1e-9 and abs(acuerdo_iso - acuerdo(np.asarray(Y_iso), pos)) < 1e-9, "Reto 3: calcula los acuerdos con acuerdo(Y, pos) sobre TUS mapas."
assert acuerdo_iso > 0.8, f"Reto 3: Isomap deberia recuperar la posicion (acuerdo > 0.8) y obtuvo {acuerdo_iso:.2f}. Revisa n_neighbors=10."
assert acuerdo_pca < 0.4, f"Reto 3: PCA no deberia desenrollar la alfombra (acuerdo < 0.4) y obtuvo {acuerdo_pca:.2f}."
assert ganador == "Isomap", "Reto 3: el ganador debe ser el texto \"Isomap\" (el de mayor acuerdo)."
print(f"✅ Reto 3 correcto | acuerdo Isomap = {acuerdo_iso:.2f} vs PCA = {acuerdo_pca:.2f} | ganador: {ganador}")

---
## Para cerrar: tu reflexión 🧠 (10 puntos)

*Sin código.* En **3 a 5 líneas** cuenta con tus palabras (y tus números): (a) ¿qué significa que 2 sombras conserven el 95 % de la información de las flores? (b) ¿por qué la sombra recta de PCA no sirve para una alfombra enrollada pero el camino de la hormiga sí?

> *(escribe aquí)*

---
### 🧠 Rúbrica corta

| Criterio | Puntos |
|---|---|
| Reto 1: la mejor sombra con PCA | 30 |
| Reto 2: cuántas sombras guardar | 30 |
| Reto 3: PCA frente a Isomap en la alfombra | 30 |
| Reflexión final | 10 |
| **Total** | **100** |

**Checklist:** ☐ Restart & Run All sin errores · ☐ los tres ✅ impresos · ☐ ningún `...` sin completar · ☐ reflexión escrita.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>